# Module 2 — Clustering

Input: `data/processed/clean_base.csv` and `feature_sets.json` from Module 1.
`sp_id` and `seller_rating_class` are **not** used to build clusters; the target is used only afterwards, as an external check.

Methods: K-Means (mandatory) + Bisecting K-Means (optional), DBSCAN (mandatory) + OPTICS (optional), and hierarchical clustering with several linkages/distances (mandatory). All stochastic steps use `random_state=42`.

In [1]:
import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram, cophenet, set_link_color_palette
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, BisectingKMeans, DBSCAN, OPTICS, cluster_optics_dbscan
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import plot_style as ps

ps.setup()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
SEED = 42

df = pd.read_csv(ROOT / "data" / "processed" / "clean_base.csv", index_col="sp_id")
fs = json.load(open(ROOT / "data" / "processed" / "feature_sets.json"))
TARGET = fs["target"]
LOG = set(fs["log1p_vars"])
print(df.shape)

(13843, 31)


## 1. Attribute selection and preprocessing

**Chosen attributes: 7 seller-profile variables** describing *how the seller does business*:

| Variable | Aspect | Transform |
|---|---|---|
| `n_listings` | size / volume | log1p |
| `price_median` | market segment | log1p |
| `savings_amount_median` | discounting | log1p |
| `daysonmarket_median` | inventory turnover | log1p |
| `mileage_median` | inventory wear | none (bimodal, log does not help; Module 1 §4.1) |
| `pct_is_new` | new/used composition | none (proportion) |
| `listing_span_days` | activity period | none (log over-corrects: skew 2.15 → −1.22) |

Why not all 19 modelling features (§1.1 shows the numbers):
- **Distance concentration:** with 19 standardized dimensions, Euclidean distances become similar, and silhouette stays around 0.2 for every k.
- **Vehicle-spec variables** (dimensions, horsepower, fuel economy, fuel tank) describe *what kind of car* the seller has, not the seller's business. They are mutually correlated (ρ_s up to 0.8), so they would dominate distances as a block. They are used afterwards to **profile** the clusters.
- **Zero-inflated proportions** (`pct_salvage` 91% zeros, `pct_theft_title` 85%) have long, sparse tails that, once standardized, turn the few non-zero sellers into artificial outliers.
- **Categorical variables** (franchise status, body type, …) are excluded from distance computation, since mixing one-hot and continuous scales needs an arbitrary weighting. They are used for profiling and as external checks.

**Preprocessing:** median imputation (fit on the full data, which is allowed for unsupervised analysis per the Module 1 leakage rule; only 1 missing value in these 7 columns) → log1p where indicated → `StandardScaler`, so every attribute contributes equally to Euclidean distances.

In [2]:
F = ["n_listings", "price_median", "savings_amount_median", "daysonmarket_median",
     "mileage_median", "pct_is_new", "listing_span_days"]
print("NaN per feature:", df[F].isna().sum().to_dict())

def make_view(cols, log=True):
    X = df[cols].copy()
    X = X.fillna(X.median())
    if log:
        for c in cols:
            if c in LOG:
                X[c] = np.log1p(X[c])
    return StandardScaler().fit_transform(X)

X = make_view(F)
sp = df[F].corr("spearman").abs().where(~np.eye(len(F), dtype=bool))
print("max |rho_s| among the 7 features:", round(sp.max().max(), 3), "(", sp.stack().idxmax(), ")")

NaN per feature: {'n_listings': 0, 'price_median': 0, 'savings_amount_median': 0, 'daysonmarket_median': 0, 'mileage_median': 1, 'pct_is_new': 0, 'listing_span_days': 0}
max |rho_s| among the 7 features: 0.861 ( ('mileage_median', 'pct_is_new') )


### 1.1 Sensitivity: feature set and log transform
K-Means (`n_init=10`) silhouette for k = 2…8, on four candidate feature sets, with and without the log transform. The silhouette is computed on all 13,843 points; `min size` is the smallest cluster.

In [3]:
SETS = {
    "ALL19 (all modelling features)": fs["numeric_model"],
    "PROFILE7 (chosen)": F,
    "PROFILE7 + 3 vehicle specs": F + ["horsepower_median", "fuel_tank_volume_median", "city_fuel_economy_median"],
}
sens = []
for name, cols in SETS.items():
    for log in [True, False]:
        Xv = make_view(cols, log)
        for k in range(2, 9):
            lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xv)
            sens.append((name, "log1p" if log else "raw", k, silhouette_score(Xv, lab), np.bincount(lab).min()))
sens = pd.DataFrame(sens, columns=["feature set", "scale", "k", "silhouette", "min size"])
sens.to_csv(ps.TAB_DIR / "m2_feature_set_sensitivity.csv", index=False)
sens_w = sens.pivot_table(index=["feature set", "scale"], columns="k", values="silhouette").round(3)
sens_m = sens.pivot_table(index=["feature set", "scale"], columns="k", values="min size")
print(sens_m)
sens_w

k                                          2       3       4       5       6      7      8
feature set                    scale                                                      
ALL19 (all modelling features) log1p  6848.0  3857.0  2636.0   914.0   120.0  117.0  119.0
                               raw    6530.0  3614.0   830.0   120.0   120.0  120.0  117.0
PROFILE7 (chosen)              log1p  6047.0  4071.0  1282.0   502.0   448.0  393.0  376.0
                               raw    6100.0   739.0   630.0   603.0   320.0   39.0   16.0
PROFILE7 + 3 vehicle specs     log1p  6015.0  2870.0  2668.0  2026.0  1086.0  893.0  400.0
                               raw    6099.0  2752.0  2341.0   555.0   489.0   49.0   58.0


k                                         2      3      4      5      6      7      8
feature set                    scale                                                 
ALL19 (all modelling features) log1p  0.220  0.189  0.174  0.169  0.181  0.179  0.169
                               raw    0.206  0.177  0.178  0.191  0.170  0.175  0.165
PROFILE7 (chosen)              log1p  0.421  0.333  0.343  0.325  0.237  0.217  0.209
                               raw    0.415  0.428  0.331  0.279  0.289  0.289  0.292
PROFILE7 + 3 vehicle specs     log1p  0.318  0.268  0.235  0.236  0.246  0.239  0.232
                               raw    0.309  0.276  0.253  0.264  0.232  0.232  0.232

Reading the table:
- **ALL19** never exceeds 0.22, a consequence of distance concentration. Adding even 3 vehicle specs lowers the silhouette at every k.
- **Raw scale** sometimes yields a *higher* silhouette (e.g. PROFILE7 raw, k = 3), but only by isolating a small group of extreme sellers (see `min size`), which is an outlier-driven partition and not a segmentation. The log view gives balanced clusters, so we keep it.

## 2. Centroid-based clustering
### 2.1 K-Means sweep (k = 2…12) and Bisecting K-Means

K-Means: `n_init=10`, k-means++ initialization. Bisecting K-Means (optional) splits the largest cluster at each step (`bisecting_strategy="largest_cluster"`). For each k we record SSE, silhouette, Davies-Bouldin (lower = better) and Calinski-Harabasz (higher = better).

In [4]:
K_RANGE = range(2, 13)
rows, km_models = [], {}
for k in K_RANGE:
    for algo, Model in [("K-Means", KMeans), ("Bisecting K-Means", BisectingKMeans)]:
        kw = dict(n_init=10, random_state=SEED)
        if algo == "Bisecting K-Means":
            kw["bisecting_strategy"] = "largest_cluster"
        m = Model(k, **kw).fit(X)
        lab = m.labels_
        rows.append((algo, k, m.inertia_, silhouette_score(X, lab), davies_bouldin_score(X, lab),
                     calinski_harabasz_score(X, lab), np.bincount(lab).min(), np.bincount(lab).max()))
        if algo == "K-Means":
            km_models[k] = m
sweep = pd.DataFrame(rows, columns=["algorithm", "k", "SSE", "silhouette", "davies_bouldin", "calinski_harabasz", "min size", "max size"])
sweep.to_csv(ps.TAB_DIR / "m2_kmeans_sweep.csv", index=False)
sweep.round(3)

,algorithm,k,SSE,silhouette,davies_bouldin,calinski_harabasz,min size,max size
0,K-Means,2,52823.574,0.421,0.953,11549.314,6047,7796
1,Bisecting K-Means,2,52823.560,0.421,0.953,11549.314,6047,7796
2,K-Means,3,42777.466,0.333,1.290,8755.428,4071,5108
3,Bisecting K-Means,3,44282.281,0.310,1.352,8222.750,3479,6047
4,K-Means,4,36131.487,0.343,1.174,7758.600,1282,5069
5,Bisecting K-Means,4,40520.743,0.260,1.570,6418.504,1212,4835
6,K-Means,5,33031.225,0.325,1.243,6689.370,502,5007
7,Bisecting K-Means,5,38166.943,0.213,1.592,5323.747,1212,4317
8,K-Means,6,30661.559,0.237,1.420,5978.535,448,3523
9,Bisecting K-Means,6,32962.227,0.225,1.516,5368.110,1179,3479


In [5]:
# Figure: SSE / silhouette / Davies-Bouldin vs k
fig, axes = plt.subplots(1, 3, figsize=(ps.FULL_W, 2.1))
for ax, metric, title in zip(axes, ["SSE", "silhouette", "davies_bouldin"],
                             ["SSE (elbow)", "Silhouette (higher = better)", "Davies-Bouldin (lower = better)"]):
    for i, algo in enumerate(["K-Means", "Bisecting K-Means"]):
        s = sweep[sweep.algorithm == algo]
        ax.plot(s.k, s[metric], marker="o", markersize=4, color=ps.CATEGORICAL[i], label=algo,
                linestyle="-" if i == 0 else "--")
    ax.axvline(4, color=ps.NEUTRAL, linewidth=1, linestyle=":")
    ax.set_title(title); ax.set_xlabel("k"); ax.set_xticks(list(K_RANGE))
    ax.tick_params(axis="x", labelsize=6)
axes[0].legend(frameon=False)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:.0f}k"))
fig.tight_layout()
ps.save(fig, "m2_kmeans_sweep")

**Choice of k = 4.**
- **k = 2** has the best silhouette, but it only splits new/franchise sellers from used-car sellers, which `new_used_profile` already gives us for free (ARI 0.83, §2.2). It adds no new information.
- **k = 4** is the best k > 2 on *both* silhouette and Davies-Bouldin (a local optimum, better than k = 3 and k = 5), its smallest cluster still holds 9% of sellers, and the SSE curve flattens after it (the elbow is soft, as expected for continuous data).
- **k ≥ 6:** the silhouette drops below 0.25, meaning that the extra clusters cut through dense regions.

Bisecting K-Means coincides with K-Means at k = 2 (it starts from the same split) but is worse for every k > 2. Its greedy, irreversible splits cannot re-assign points between the two halves, while K-Means re-optimizes all centroids jointly.

In [6]:
# Stability of the chosen configuration across seeds
K = 4
base = KMeans(K, n_init=10, random_state=SEED).fit(X)
aris = [adjusted_rand_score(base.labels_, KMeans(K, n_init=10, random_state=s).fit_predict(X)) for s in range(10)]
print("ARI of k=4 across 10 seeds: min %.4f, mean %.4f" % (min(aris), np.mean(aris)))

ARI of k=4 across 10 seeds: min 0.9981, mean 0.9996


### 2.2 K-Means (k = 4): cluster profiles
Clusters are renumbered deterministically by decreasing median price, so the numbering does not depend on the random initialization.

In [7]:
def relabel_by(labels, key, ascending=False):
    """Renumber clusters 0..k-1 ordered by the median of `key`; negative labels (noise) are kept."""
    s = pd.Series(labels, index=df.index)
    order = df[key].groupby(s[s >= 0]).median().sort_values(ascending=ascending).index
    mapping = {old: new for new, old in enumerate(order)}
    return s.map(lambda v: mapping.get(v, v)).to_numpy()

km_lab = relabel_by(base.labels_, "price_median")
PROFILE_COLS = ["n_listings", "price_median", "savings_amount_median", "daysonmarket_median", "mileage_median",
                "pct_is_new", "listing_span_days", "vehicle_age_median", "horsepower_median", "pct_isCab", "pct_salvage"]

def profile(labels, name):
    s = pd.Series(labels, index=df.index, name=name)
    p = df[PROFILE_COLS].groupby(s).median()
    p.insert(0, "size", s.value_counts().sort_index())
    p.insert(1, "size %", (p["size"] / len(s) * 100).round(1))
    p["franchise %"] = (df.franchise_status.eq("franchise").groupby(s).mean() * 100).round(1)
    p["top body type"] = df.dominant_body_type.groupby(s).agg(lambda x: x.value_counts().index[0])
    p["suv %"] = (df.dominant_body_type.eq("suv_crossover").groupby(s).mean() * 100).round(1)
    r = pd.crosstab(s, df[TARGET], normalize="index")[ps.CLASS_ORDER] * 100
    r.columns = [f"{c} %" for c in r.columns]
    return pd.concat([p, r.round(1)], axis=1)

km_prof = profile(km_lab, "kmeans4")
km_prof.to_csv(ps.TAB_DIR / "m2_kmeans4_profiles.csv")
km_prof.round(2)

,size,size %,n_listings,price_median,savings_amount_median,daysonmarket_median,mileage_median,pct_is_new,listing_span_days,vehicle_age_median,horsepower_median,pct_isCab,pct_salvage,franchise %,top body type,suv %,low %,medium %,high %
kmeans4,,,,,,,,,,,,,,,,,,,
0,5069,36.6,185.0,30059.0,0.00,38.5,11.5,0.65,418.0,0.0,250.0,0.12,0.0,99.7,suv_crossover,80.1,22.4,57.8,19.8
1,3658,26.4,71.0,22990.0,703.00,32.0,33299.0,0.00,322.0,3.0,252.0,0.13,0.0,48.9,suv_crossover,68.8,16.4,58.6,25.0
2,1282,9.3,32.0,12500.0,615.75,92.0,94946.0,0.00,733.0,7.0,240.0,0.12,0.0,10.5,suv_crossover,47.4,19.4,44.9,35.7
3,3834,27.7,19.0,9745.0,589.25,34.0,112901.5,0.00,204.0,8.5,215.0,0.11,0.0,0.6,suv_crossover,45.3,15.5,50.3,34.2


In [8]:
def external(labels, mask=None):
    m = np.ones(len(labels), bool) if mask is None else mask
    return {"ARI rating": adjusted_rand_score(df[TARGET][m], labels[m]),
            "NMI rating": normalized_mutual_info_score(df[TARGET][m], labels[m]),
            "ARI franchise": adjusted_rand_score(df.franchise_status[m], labels[m]),
            "ARI new_used_profile": adjusted_rand_score(df.new_used_profile[m], labels[m])}

k2_lab = km_models[2].labels_
print("k=2:", {k: round(v, 3) for k, v in external(k2_lab).items()})
print("k=4:", {k: round(v, 3) for k, v in external(km_lab).items()})

k=2: {'ARI rating': 0.006, 'NMI rating': 0.01, 'ARI franchise': 0.699, 'ARI new_used_profile': 0.829}
k=4: {'ARI rating': 0.014, 'NMI rating': 0.011, 'ARI franchise': 0.425, 'ARI new_used_profile': 0.434}


In [9]:
# Figure: standardized centroid profile (z-scores of the clustering features) + rating mix per cluster
NAMES = {0: "C0", 1: "C1", 2: "C2", 3: "C3"}
Z = pd.DataFrame(X, columns=F, index=df.index).groupby(km_lab).mean()
short = {"n_listings": "n_listings*", "price_median": "price*", "savings_amount_median": "savings*",
         "daysonmarket_median": "days on mkt*", "mileage_median": "mileage", "pct_is_new": "pct_is_new",
         "listing_span_days": "listing span"}
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.2), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
im = ax.imshow(Z.values, cmap=ps.DIVERGING, vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(F))); ax.set_xticklabels([short[c] for c in F], rotation=30, ha="right", fontsize=6.5)
ax.set_yticks(range(K)); ax.set_yticklabels([f"{NAMES[i]} (n={km_prof.loc[i, 'size']:,})" for i in range(K)], fontsize=6.5)
for i in range(K):
    for j in range(len(F)):
        v = Z.values[i, j]
        ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=6, color="white" if abs(v) > 1.1 else ps.INK)
ax.grid(False)
for s_ in ax.spines.values(): s_.set_visible(False)
cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02); cb.set_label("centroid z-score", fontsize=6.5); cb.outline.set_visible(False)
ax.set_title("K-Means k=4 centroids (* = log1p)")

ax = axes[1]
rmix = pd.crosstab(km_lab, df[TARGET], normalize="index")[ps.CLASS_ORDER]
left = np.zeros(K)
for c in ps.CLASS_ORDER:
    ax.barh([NAMES[i] for i in range(K)], rmix[c], left=left, color=ps.CLASS_COLORS[c], label=c, height=0.6,
            edgecolor="white", linewidth=1.5)
    for y in range(K):
        ax.text(left[y] + rmix[c].iloc[y] / 2, y, f"{rmix[c].iloc[y]:.0%}", ha="center", va="center", fontsize=6,
                color=ps.INK if c == "low" else "white")
    left += rmix[c].values
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("share of sellers")
ax.grid(axis="y", visible=False)
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False, fontsize=6.5,
          title="rating mix per cluster", title_fontsize=7.5)
fig.tight_layout()
ps.save(fig, "m2_kmeans4_profile")

In [10]:
# Figure: the clusters in two interpretable original dimensions (small multiples: one cluster highlighted per panel,
# because 4 overlapping categorical colors in one scatter are not reliably distinguishable)
fig, axes = plt.subplots(1, K, figsize=(ps.FULL_W, 1.9), sharex=True, sharey=True)
xv = np.log1p(df.n_listings); yv = df.mileage_median / 1000
for i, ax in enumerate(axes):
    m = km_lab == i
    ax.scatter(xv[~m], yv[~m], s=1, color=ps.GRID, rasterized=True)
    ax.scatter(xv[m], yv[m], s=1, color=ps.CATEGORICAL[0], alpha=0.5, rasterized=True)
    ax.set_title(f"{NAMES[i]} ({m.mean():.0%})", fontsize=7.5)
    ax.set_xlabel("log1p n_listings", fontsize=6.5)
axes[0].set_ylabel("median mileage (k mi)")
fig.tight_layout()
ps.save(fig, "m2_kmeans4_scatter")

**Interpretation of the four K-Means segments:**
- **C0, franchise new-car dealers:** large volume, about 65% new cars, near-zero mileage, high prices, no advertised savings; almost all are franchise dealers.
- **C1, mid-market used dealers:** mostly late-model used cars (≈ 33k miles), mid prices, regular discounting; half franchise, half independent.
- **C2, slow-turnover used dealers:** older, high-mileage stock that stays listed about 3× longer (days on market ≈ 90) over a much longer activity span; mostly independent.
- **C3, small budget independents:** very few listings, old high-mileage cars at the lowest prices, short activity span.

**Clusters vs. rating.** The rating is not what separates the sellers: ARI with `seller_rating_class` ≈ 0.01. The rating mix still **shifts with seller type**: the share of `high` rises from ≈ 20% in C0 (franchise) to 25% in C1 and ≈ 34–36% in the independent segments C2/C3. This repeats the Module 1 finding that small independent sellers are rated higher, but each segment still contains every rating level, consistent with a "hard" classification task.

## 3. Density-based clustering
### 3.1 DBSCAN: choosing MinPts and eps

- **MinPts:** the common heuristic MinPts ≈ 2·d gives 14 for d = 7; we test {5, 10, 15, 20, 30} around it.
- **eps:** read from the **k-distance graph** (distance of each point to its MinPts-th neighbour, sorted). The tested eps range (0.35–1.0) covers the region where the curves bend.
- **Dimensionality:** we keep the same 7 standardized features. With more dimensions, distances concentrate (§1.1) and a single global eps becomes even harder to set.

In [11]:
MINPTS = [5, 10, 15, 20, 30]
EPS_PLOT = 0.6  # the eps selected by the rule in the next cells (asserted there), marked on the plot
EPS = [0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.8, 1.0]
kdist = {}
nn = NearestNeighbors(n_neighbors=max(MINPTS)).fit(X)
dist, _ = nn.kneighbors(X)  # column 0 is the point itself (distance 0), as in DBSCAN's min_samples counting
for mp in MINPTS:
    kdist[mp] = np.sort(dist[:, mp - 1])

fig, ax = plt.subplots(figsize=(ps.COL_W, 2.3))
for i, mp in enumerate(MINPTS):
    ax.plot(np.arange(len(X)), kdist[mp], color=ps.CATEGORICAL[i] if i < 3 else [ps.CATEGORICAL[3], ps.CATEGORICAL[4]][i - 3],
            linewidth=1.5, label=f"MinPts={mp}")
ax.axhspan(0.35, 1.0, color=ps.GRID, alpha=0.5, zorder=0)
ax.text(300, 0.97, "tested eps range", fontsize=6.5, color=ps.INK_2, va="top")
ax.axhline(EPS_PLOT, color=ps.INK_2, linewidth=0.8, linestyle="--")
ax.text(300, EPS_PLOT + 0.03, f"chosen eps = {EPS_PLOT}", fontsize=6.5, color=ps.INK_2, va="bottom")
ax.set_ylim(0, 2.5); ax.set_xlabel("points sorted by k-distance"); ax.set_ylabel("distance to MinPts-th neighbour")
ax.legend(frameon=False, fontsize=6, loc="upper left", bbox_to_anchor=(0, 0.88))
ax.set_title("k-distance graphs")
fig.tight_layout()
ps.save(fig, "m2_kdistance")
print({mp: np.round(np.quantile(kdist[mp], [.5, .8, .9, .95]), 2).tolist() for mp in MINPTS})

{5: [0.42, 0.61, 0.78, 0.98], 10: [0.49, 0.71, 0.91, 1.14], 15: [0.53, 0.77, 0.99, 1.24], 20: [0.57, 0.82, 1.06, 1.33], 30: [0.62, 0.88, 1.15, 1.46]}


In [12]:
grid = []
for mp in MINPTS:
    for eps in EPS:
        lab = DBSCAN(eps=eps, min_samples=mp).fit_predict(X)
        core = lab >= 0
        n_cl = lab.max() + 1
        sizes = np.sort(np.bincount(lab[core]))[::-1] if n_cl else np.array([])
        sil = silhouette_score(X[core], lab[core]) if n_cl >= 2 else np.nan
        grid.append((mp, eps, n_cl, int((sizes >= 0.01 * len(X)).sum()), (~core).mean() * 100, sil,
                     ", ".join(map(str, sizes[:3])) + (" …" if n_cl > 3 else "")))
grid = pd.DataFrame(grid, columns=["MinPts", "eps", "clusters", "clusters >= 1%", "noise %", "silhouette (non-noise)", "largest sizes"])
grid.to_csv(ps.TAB_DIR / "m2_dbscan_grid.csv", index=False)
grid.round(3)

,MinPts,eps,clusters,clusters >= 1%,noise %,silhouette (non-noise),largest sizes
0,5,0.35,66,3,55.371,-0.319,"3115, 2504, 163 …"
1,5,0.40,45,3,41.797,-0.278,"3828, 3593, 400 …"
2,5,0.45,31,2,31.272,-0.232,"5437, 3909, 14 …"
3,5,0.50,23,2,23.333,-0.142,"6312, 4174, 13 …"
4,5,0.55,9,1,17.966,-0.204,"11301, 10, 10 …"
5,5,0.60,21,1,13.993,-0.366,"11792, 8, 8 …"
6,5,0.70,9,1,8.734,-0.187,"12534, 43, 16 …"
7,5,0.80,7,1,5.873,-0.129,"12973, 33, 6 …"
8,5,1.00,4,1,2.644,0.081,"13454, 12, 6 …"
9,10,0.35,17,3,71.357,0.150,"2576, 732, 441 …"


**Selection rule:** among configurations with **at least two clusters each holding ≥ 1% of the sellers**, pick the one with the **lowest noise**, with ties broken by silhouette.

Larger eps values merge everything into one cluster: this happens at the knee of the k-distance curves (eps ≈ 0.7–1.0), which is the classic choice, so that choice would yield a single cluster. Smaller eps fragments the data into many micro-clusters and labels half of the sellers as noise.

In [13]:
valid = grid[grid["clusters >= 1%"] >= 2].sort_values(["noise %", "silhouette (non-noise)"], ascending=[True, False])
print(valid.head(6).round(3).to_string())
best = valid.iloc[0]
EPS_STAR, MP_STAR = float(best.eps), int(best.MinPts)
assert EPS_STAR == EPS_PLOT, "update EPS_PLOT in the k-distance cell"
db = DBSCAN(eps=EPS_STAR, min_samples=MP_STAR).fit(X)
# renumber clusters by size (largest = 0); noise stays -1
sizes = pd.Series(db.labels_[db.labels_ >= 0]).value_counts()
db_lab = np.array([{old: new for new, old in enumerate(sizes.index)}.get(v, -1) for v in db.labels_])
print(f"chosen: eps={EPS_STAR}, MinPts={MP_STAR}; sizes:", pd.Series(db_lab).value_counts().sort_index().to_dict())

    MinPts   eps  clusters  clusters >= 1%  noise %  silhouette (non-noise)     largest sizes
23      15  0.60         2               2   21.997                   0.493        6542, 4256
3        5  0.50        23               2   23.333                  -0.142  6312, 4174, 13 …
13      10  0.55         6               2   23.969                   0.155  6265, 4194, 36 …
32      20  0.60         2               2   25.038                   0.505        6186, 4191
22      15  0.55         2               2   28.505                   0.516        5821, 4076
41      30  0.60         2               2   29.307                   0.522        5690, 4096


chosen: eps=0.6, MinPts=15; sizes: {-1: 3045, 0: 6542, 1: 4256}


In [14]:
db_prof = profile(db_lab, "dbscan")
db_prof.index = ["noise" if i == -1 else f"D{i}" for i in db_prof.index]
db_prof.to_csv(ps.TAB_DIR / "m2_dbscan_profiles.csv")
print(pd.crosstab(db_lab, km_lab, rownames=["DBSCAN"], colnames=["K-Means k=4"]))
db_prof.round(2)

K-Means k=4     0     1    2     3
DBSCAN                            
-1            681   680  993   691
 0            132  2978  289  3143
 1           4256     0    0     0


,size,size %,n_listings,price_median,savings_amount_median,daysonmarket_median,mileage_median,pct_is_new,listing_span_days,vehicle_age_median,horsepower_median,pct_isCab,pct_salvage,franchise %,top body type,suv %,low %,medium %,high %
noise,3045,22.0,39.0,19888.00,370.00,49.5,49752.5,0.00,520.0,4.0,248.0,0.10,0.0,37.9,suv_crossover,53.8,19.2,48.2,32.6
D0,6542,47.3,37.0,15239.50,644.75,34.0,79225.0,0.00,265.0,5.5,240.0,0.12,0.0,24.3,suv_crossover,59.5,15.8,55.4,28.7
D1,4256,30.7,193.0,29822.25,0.00,37.0,11.0,0.66,407.0,0.0,250.0,0.12,0.0,100.0,suv_crossover,79.7,22.5,58.3,19.1


In [15]:
# What makes the noise dealers anomalous? Share of noise vs. core sellers with each extreme trait
noise = db_lab == -1
traits = {
    "n_listings > p95": df.n_listings > df.n_listings.quantile(.95),
    "n_listings <= 10": df.n_listings <= 10,
    "days on market > 120": df.daysonmarket_median > 120,
    "listing span > p95": df.listing_span_days > df.listing_span_days.quantile(.95),
    "price > p95": df.price_median > df.price_median.quantile(.95),
    "price < p5": df.price_median < df.price_median.quantile(.05),
    "0 < pct_is_new < 0.2 (few new cars)": df.pct_is_new.between(0, 0.2, inclusive="neither"),
    "new_only seller": df.new_used_profile.eq("new_only"),
    "mixed seller with mileage > 20k": df.new_used_profile.eq("mixed") & (df.mileage_median > 20000),
    "any trait above": None,
}
any_t = np.zeros(len(df), bool)
for k_, v in traits.items():
    if v is not None:
        any_t |= v.to_numpy()
traits["any trait above"] = pd.Series(any_t, index=df.index)
nt = pd.DataFrame({"noise %": {k_: v[noise].mean() * 100 for k_, v in traits.items()},
                   "core %": {k_: v[~noise].mean() * 100 for k_, v in traits.items()}}).round(1)
nt["ratio"] = (nt["noise %"] / nt["core %"].replace(0, np.nan)).round(1)
nt.to_csv(ps.TAB_DIR / "m2_dbscan_noise_traits.csv")
nt

,noise %,core %,ratio
n_listings > p95,4.0,5.2,0.8
n_listings <= 10,16.5,5.1,3.2
days on market > 120,22.7,0.3,75.7
listing span > p95,20.8,0.5,41.6
price > p95,10.5,3.4,3.1
price < p5,11.9,3.1,3.8
0 < pct_is_new < 0.2 (few new cars),4.1,2.3,1.8
new_only seller,0.7,0.1,7.0
mixed seller with mileage > 20k,12.1,8.5,1.4
any trait above,73.2,24.9,2.9


**DBSCAN result.** The density structure has **two dense cores**, the same split as K-Means k = 2:
- **D0:** used-car dealers, which absorbs most of K-Means C1 and C3.
- **D1:** franchise new/mixed dealers, which matches K-Means C0.

**Noise (22%)** is not made of corrupted records. The traits table shows which traits are over-represented among noise points (the "ratio" column):
- **Slow turnover dominates:** a median days-on-market above 120 is 76× more frequent among noise points, and a listing span above the 95th percentile 42× more frequent. 77% of K-Means cluster C2 (slow-turnover dealers) is labelled noise.
- **Weaker signals** (ratio ≈ 3): very small sellers (≤ 10 listings) and extreme price points (top or bottom 5%).
- **Not over-represented:** large sellers (ratio 0.8). Transitional profiles, such as mixed dealers with mostly used stock, are only mildly over-represented (ratio 1.4–1.8).

DBSCAN uses a single global density threshold. So when two modes are joined by a continuum and a sparse tail, it can only separate them by discarding both the tail and the boundary region.

### 3.2 Outlier sensitivity of K-Means
K-Means is re-fit (k = 4) on the DBSCAN core points only, and the new centroids are used to assign *all* sellers. A high ARI with the original partition means that the noise/outlier sellers were not driving the K-Means centroids.

In [16]:
km_core = KMeans(K, n_init=10, random_state=SEED).fit(X[~noise])
km_core_all = km_core.predict(X)
print("ARI(K-Means on all points, K-Means fit on DBSCAN core only):", round(adjusted_rand_score(km_lab, km_core_all), 3))
print(pd.crosstab(km_lab, km_core_all, rownames=["original k=4"], colnames=["fit on core only"]))

ARI(K-Means on all points, K-Means fit on DBSCAN core only): 0.789


fit on core only     0     1     2     3
original k=4                            
0                    0  5061     8     0
1                   49     1  3267   341
2                    0    56    92  1134
3                 2414     0     1  1419


The ARI of 0.79 shows **moderate** sensitivity, and the crosstab shows where it comes from:
- **C0 and C1 survive almost unchanged.**
- **C2 (slow turnover) disappears:** most of its members are DBSCAN noise, so without them K-Means spends the fourth centroid on splitting the budget segment C3 in two, and C2's sellers merge into one of the two halves.

C2 is therefore a real but *sparse* segment, defined by its tail behaviour (long listing spans, slow sales), rather than a dense group. This caveat is reported alongside the k = 4 interpretation.

### 3.3 OPTICS (optional): reachability structure
OPTICS with the same MinPts orders the points so that density valleys become visible as dips in the reachability plot, without fixing a single eps.

In [17]:
t0 = time.time()
op = OPTICS(min_samples=MP_STAR, xi=0.05).fit(X)
print("OPTICS fit: %.1fs" % (time.time() - t0), "| xi-clusters found:", op.labels_.max() + 1)
reach = op.reachability_[op.ordering_]
op_db = cluster_optics_dbscan(reachability=op.reachability_, core_distances=op.core_distances_,
                              ordering=op.ordering_, eps=EPS_STAR)
print("ARI(OPTICS cut at eps*, DBSCAN):", round(adjusted_rand_score(db_lab, op_db), 3))

fig, ax = plt.subplots(figsize=(ps.COL_W, 2.0))
r = np.where(np.isinf(reach), np.nan, reach)
lab_ord = db_lab[op.ordering_]
colors = np.where(lab_ord == -1, ps.NEUTRAL, np.where(lab_ord == 0, ps.CATEGORICAL[0], ps.CATEGORICAL[1]))
ax.bar(np.arange(len(r)), r, width=1.0, color=colors, linewidth=0)
ax.axhline(EPS_STAR, color=ps.INK_2, linestyle="--", linewidth=0.8)
ax.text(len(r) * 0.62, EPS_STAR + 0.04, f"eps = {EPS_STAR}", ha="center", fontsize=6.5, color=ps.INK_2)
ax.set_ylim(0, 2.0); ax.set_xlabel("OPTICS ordering"); ax.set_ylabel("reachability distance")
ax.legend(handles=[Patch(color=ps.CATEGORICAL[0], label="D0"), Patch(color=ps.CATEGORICAL[1], label="D1"),
                   Patch(color=ps.NEUTRAL, label="noise")], frameon=False, ncol=3, fontsize=6.5, loc="upper left")
ax.grid(axis="x", visible=False)
fig.tight_layout()
ps.save(fig, "m2_optics_reachability")

OPTICS fit: 12.7s | xi-clusters found: 1
ARI(OPTICS cut at eps*, DBSCAN): 0.999


The reachability plot shows **two broad, shallow valleys** (the two DBSCAN cores), separated by a short rise, followed by a long, steadily rising tail (the noise). There are no steep walls, so the ξ-method (ξ = 0.05), which looks for steep drops in reachability, returns a **single cluster containing all sellers**. Cutting the same ordering at eps = 0.6 reproduces DBSCAN almost exactly (ARI 0.999). This is direct evidence that the dataset has **uniform, continuous density** with two broad modes rather than well-separated blobs, and that a density-based method is not the natural fit for this geometry.

## 4. Hierarchical clustering

Computed on **all 13,843 sellers** (no sampling; the full distance matrix fits in memory), on the same 7-feature standardized view.

Distance/linkage combinations:
- **Euclidean** with Single, Complete, Average, and Ward. Ward is only defined for Euclidean distances, since it minimizes the increase in within-cluster variance.
- **Manhattan** with Complete and Average: more robust to large differences on a single attribute.
- **Cosine** with Average: it compares *profile shape*, ignoring magnitude.

For each combination we report the cophenetic correlation (how faithfully the dendrogram preserves the original distances) and the silhouette (computed with the same metric) for cuts into k = 2…6 clusters.

In [18]:
COMBOS = [("single", "euclidean"), ("complete", "euclidean"), ("average", "euclidean"), ("ward", "euclidean"),
          ("complete", "cityblock"), ("average", "cityblock"), ("average", "cosine")]
SIL_METRIC = {"euclidean": "euclidean", "cityblock": "manhattan", "cosine": "cosine"}
links, hrows = {}, []
for meth, metric in COMBOS:
    t0 = time.time()
    Zl = linkage(X, method=meth, metric=metric)
    links[(meth, metric)] = Zl
    coph = cophenet(Zl, __import__("scipy.spatial.distance", fromlist=["pdist"]).pdist(X, metric=metric))[0]
    for k in range(2, 7):
        lab = fcluster(Zl, k, criterion="maxclust")
        cnt = np.bincount(lab)[1:]
        sil = silhouette_score(X, lab, metric=SIL_METRIC[metric]) if len(cnt) > 1 else np.nan
        hrows.append((meth, metric, round(coph, 3), k, sil, cnt.min(), cnt.max()))
    print(f"{meth}/{metric}: {time.time() - t0:.1f}s")
hier = pd.DataFrame(hrows, columns=["linkage", "metric", "cophenetic r", "k", "silhouette", "min size", "max size"])
hier.to_csv(ps.TAB_DIR / "m2_hierarchical.csv", index=False)
hier.pivot_table(index=["linkage", "metric", "cophenetic r"], columns="k", values=["silhouette", "min size"]).round(3)

single/euclidean: 14.9s


complete/euclidean: 18.3s


average/euclidean: 18.8s


ward/euclidean: 20.2s


complete/cityblock: 21.2s


average/cityblock: 17.9s


average/cosine: 18.7s


min size                              silhouette                            
k                                      2       3      4      5      6          2      3      4      5      6
linkage  metric    cophenetic r                                                                             
average  cityblock 0.755             2.0     2.0    1.0    1.0    1.0      0.601  0.339  0.338  0.444  0.439
         cosine    0.855          5915.0   153.0  153.0    2.0    2.0      0.621  0.485  0.470  0.422  0.371
         euclidean 0.761             1.0     1.0    1.0    1.0    1.0      0.710  0.429  0.376  0.322  0.321
complete cityblock 0.642           222.0   222.0  222.0   27.0   27.0      0.295  0.250  0.326  0.321  0.263
         euclidean 0.533           268.0   268.0    6.0    6.0    6.0      0.381  0.157  0.156  0.144  0.073
single   euclidean 0.509             1.0     1.0    1.0    1.0    1.0      0.710  0.657  0.531  0.525  0.429
ward     euclidean 0.648          4819.0  4108.0  762.0  656.0  656.0      0.399  0.322  0.318  0.306  0.264

In [19]:
# Figure: six dendrograms (truncated to the last 30 merges; leaf labels = cluster sizes are omitted for legibility)
set_link_color_palette(ps.CATEGORICAL[:4])
SHOW = [("single", "euclidean"), ("complete", "euclidean"), ("average", "euclidean"),
        ("ward", "euclidean"), ("average", "cityblock"), ("average", "cosine")]
fig, axes = plt.subplots(2, 3, figsize=(ps.FULL_W, 3.8))
for ax, key in zip(axes.ravel(), SHOW):
    Zl = links[key]
    cut = (Zl[-4, 2] + Zl[-3, 2]) / 2  # height that yields 4 clusters
    dendrogram(Zl, truncate_mode="lastp", p=30, ax=ax, no_labels=True, color_threshold=cut,
               above_threshold_color=ps.NEUTRAL)
    ax.axhline(cut, color=ps.INK_2, linestyle="--", linewidth=0.8)
    lab4 = fcluster(Zl, 4, criterion="maxclust")
    sizes4 = sorted(np.bincount(lab4)[1:], reverse=True)
    name = {"cityblock": "manhattan"}.get(key[1], key[1])
    ax.set_title(f"{key[0]} / {name}\nk=4 sizes: {', '.join(map(str, sizes4))}", fontsize=7)
    ax.grid(axis="x", visible=False); ax.tick_params(axis="y", labelsize=6)
axes[0, 0].set_ylabel("merge distance"); axes[1, 0].set_ylabel("merge distance")
fig.tight_layout()
ps.save(fig, "m2_dendrograms")
set_link_color_palette(None)

**Structural phenomena in the dendrograms:**
- **Single linkage** shows pure *chaining*: points join one by one at low heights. Every cut produces one giant cluster plus singletons, so a high silhouette there is an artefact of isolating a few outliers.
- **Average** linkage (Euclidean and Manhattan) has the same problem in milder form: the top merges separate tiny groups of extreme sellers.
- **Complete** linkage avoids singletons but is still driven by extremes, because the maximum pairwise distance decides each merge. Its k = 4 cut puts 89% of sellers in one cluster and has the lowest cophenetic correlation of the non-single linkages (0.53). The Manhattan variant is somewhat more balanced (smallest cluster 222 sellers).
- **Ward** produces compact, balanced clusters with long vertical branches at the top. These are the clearest cut points, and Ward also matches the objective of K-Means.
- **Cosine/average** has the highest cophenetic correlation (0.86) and two balanced groups at k = 2. It splits sellers by the *direction* of their profile (above vs. below average on the size/newness block) and ignores how *far* a seller is from the average profile. That makes the clusters harder to interpret in original units, and its k = 4 cut adds a 153-seller fringe group.

**Chosen: Ward/Euclidean cut at k = 4.** The Ward merge heights for the transitions 5→4→3→2→1 are 76, 105, 152 and 286. The largest gap is the first split (new vs. used, k = 2). The gap above the 4-cluster level (76 → 105) is still about twice the next one down (63 → 76), so k = 4 is the last level with a clear gap. Its silhouette (0.32) is close to that of K-Means and its clusters are reasonably balanced (smallest = 5.5%).

In [20]:
ward4 = relabel_by(fcluster(links[("ward", "euclidean")], 4, criterion="maxclust") - 1, "price_median")
ward_prof = profile(ward4, "ward4")
ward_prof.index = [f"W{i}" for i in ward_prof.index]
ward_prof.to_csv(ps.TAB_DIR / "m2_ward4_profiles.csv")
print(pd.crosstab(ward4, km_lab, rownames=["Ward k=4"], colnames=["K-Means k=4"]))
print("ARI(Ward4, KMeans4):", round(adjusted_rand_score(ward4, km_lab), 3))
print("Ward merge heights, transitions k=8..1:", np.round(links[("ward", "euclidean")][-8:, 2], 1))
ward_prof.round(2)

K-Means k=4     0     1    2     3
Ward k=4                          
0            4809     1    9     0
1             233  3011   39    63
2              27   179  556     0
3               0   467  678  3771
ARI(Ward4, KMeans4): 0.756
Ward merge heights, transitions k=8..1: [ 52.5  58.5  58.8  62.9  76.4 104.8 151.8 285.6]


,size,size %,n_listings,price_median,savings_amount_median,daysonmarket_median,mileage_median,pct_is_new,listing_span_days,vehicle_age_median,horsepower_median,pct_isCab,pct_salvage,franchise %,top body type,suv %,low %,medium %,high %
W0,4819,34.8,187.0,30092.0,0.00,39.0,11.00,0.66,414.0,0.0,250.00,0.12,0.0,100.0,suv_crossover,79.8,22.7,57.4,19.8
W1,3346,24.2,77.0,23995.0,641.00,31.0,31312.25,0.00,327.0,3.0,252.00,0.12,0.0,55.5,suv_crossover,70.7,16.5,59.2,24.3
W2,762,5.5,52.0,18995.0,627.50,62.5,51419.75,0.00,948.5,4.0,251.75,0.14,0.0,31.9,suv_crossover,57.1,18.1,50.5,31.4
W3,4916,35.5,21.0,10296.0,617.75,36.0,108885.00,0.00,241.0,8.0,222.00,0.11,0.0,1.7,suv_crossover,46.2,16.1,49.7,34.2


## 5. Comparison and final discussion

In [21]:
def internal(labels, mask=None):
    m = np.ones(len(labels), bool) if mask is None else mask
    return silhouette_score(X[m], labels[m]), davies_bouldin_score(X[m], labels[m])

def sse(labels, mask=None):
    m = np.ones(len(labels), bool) if mask is None else mask
    return sum(((X[m][labels[m] == c] - X[m][labels[m] == c].mean(0)) ** 2).sum() for c in np.unique(labels[m]))

bis4 = relabel_by(BisectingKMeans(4, n_init=10, random_state=SEED, bisecting_strategy="largest_cluster").fit_predict(X), "price_median")
compl4 = fcluster(links[("complete", "euclidean")], 4, criterion="maxclust") - 1
single4 = fcluster(links[("single", "euclidean")], 4, criterion="maxclust") - 1
core = db_lab >= 0
methods = [
    ("K-Means", "k=2", km_models[2].labels_, None),
    ("K-Means", "k=4", km_lab, None),
    ("Bisecting K-Means", "k=4", bis4, None),
    ("DBSCAN", f"eps={EPS_STAR}, MinPts={MP_STAR}", db_lab, core),
    ("Hierarchical", "Ward/Euclidean, k=4", ward4, None),
    ("Hierarchical", "Complete/Euclidean, k=4", compl4, None),
    ("Hierarchical", "Single/Euclidean, k=4", single4, None),
]
comp = []
for name, params, lab, m in methods:
    s, d = internal(lab, m)
    cnt = np.bincount(lab[lab >= 0])
    ext = external(lab, m)
    comp.append({"method": name, "parameters": params, "clusters": len(cnt),
                 "noise %": round((lab < 0).mean() * 100, 1), "silhouette": s, "Davies-Bouldin": d, "SSE": sse(lab, m),
                 "min size %": cnt.min() / len(lab) * 100, "max size %": cnt.max() / len(lab) * 100,
                 "ARI vs K-Means4": adjusted_rand_score(km_lab[m if m is not None else slice(None)], lab[m if m is not None else slice(None)]),
                 **ext})
comp = pd.DataFrame(comp)
comp.to_csv(ps.TAB_DIR / "m2_comparison.csv", index=False)
comp.round(3)

,method,parameters,clusters,noise %,silhouette,Davies-Bouldin,SSE,min size %,max size %,ARI vs K-Means4,ARI rating,NMI rating,ARI franchise,ARI new_used_profile
0,K-Means,k=2,2,0.0,0.421,0.953,52823.558,43.683,56.317,0.460,0.006,0.010,0.699,0.829
1,K-Means,k=4,4,0.0,0.343,1.174,36131.487,9.261,36.618,1.000,0.014,0.011,0.425,0.434
2,Bisecting K-Means,k=4,4,0.0,0.260,1.570,40520.705,8.755,34.927,0.722,0.004,0.010,0.466,0.504
3,DBSCAN,"eps=0.6, MinPts=15",2,22.0,0.493,0.743,25565.654,30.745,47.259,0.576,0.005,0.009,0.497,0.624
4,Hierarchical,"Ward/Euclidean, k=4",4,0.0,0.318,1.212,39098.036,5.505,35.513,0.756,0.007,0.010,0.480,0.484
5,Hierarchical,"Complete/Euclidean, k=4",4,0.0,0.156,2.029,88100.544,0.043,88.579,0.046,0.024,0.007,0.002,-0.002
6,Hierarchical,"Single/Euclidean, k=4",4,0.0,0.531,0.290,96610.685,0.007,99.978,0.000,0.000,0.000,0.000,-0.000


Notes on the comparison:
- For DBSCAN, the internal metrics, SSE and ARIs are computed on **non-noise points only**, so its silhouette is optimistic: the hardest 20% of sellers have been removed. The noise percentage must be read alongside it.
- SSE is only comparable between methods with the same k.

**Which algorithm is best for this dataset?**

1. **Geometry.** The seller-profile space is a *continuum with two broad density modes* (new/franchise vs. used/independent) and a sparse slow-turnover tail, not a set of separated blobs. The evidence:
   - the soft elbow and silhouettes ≤ 0.42;
   - an OPTICS reachability plot without steep valleys (ξ finds a single cluster);
   - DBSCAN needing 22% noise to separate the two modes.
2. **Centroid-based (K-Means k = 4): best overall.**
   - It covers every seller (no noise) and has the best internal metrics among the non-degenerate k = 4 partitions. Single linkage scores higher only by isolating 3 sellers.
   - It is stable across seeds (ARI ≈ 1), and its four segments are interpretable in business terms.
   - Caveat: the slow-turnover segment C2 depends on the sparse tail and disappears when DBSCAN noise is removed (ARI 0.79, §3.2).
   - Its assumption of convex, similar-variance clusters is acceptable because the clusters are *partitions of a continuum* rather than natural shapes.
3. **Hierarchical Ward** largely agrees with K-Means (ARI 0.76; same variance objective) and adds the dendrogram as a view of how segments nest: the first and largest split is again new vs. used. Single and average linkage fail through chaining and singletons; complete linkage is driven by extremes.
4. **DBSCAN/OPTICS** are the wrong tool for *segmentation* here, but useful for *anomaly profiling*: the noise sellers are mainly the slow-turnover, long-span inventories, plus very small sellers and extreme price points.
5. **External check:** no method recovers `seller_rating_class` (ARI ≤ 0.025). Seller type only shifts the rating mix (more `high` among small independents), which anticipates the low ceiling of the Module 3 classifiers.

In [22]:
# Save cluster labels for later modules (profiling with M4 rules, cross-module discussion)
labels_out = pd.DataFrame({"kmeans4": km_lab, "kmeans2": km_models[2].labels_, "ward4": ward4, "dbscan": db_lab},
                          index=df.index)
labels_out.to_csv(ROOT / "data" / "processed" / "cluster_labels.csv")
labels_out.apply(lambda s: s.value_counts().sort_index().to_dict())

kmeans4    {0: 5069, 1: 3658, 2: 1282, 3: 3834}
kmeans2                      {0: 6047, 1: 7796}
ward4       {0: 4819, 1: 3346, 2: 762, 3: 4916}
dbscan             {-1: 3045, 0: 6542, 1: 4256}
dtype: object